# Veraxi RAG + Code-Intel Evaluation on Google Colab Pro (Zero External API Keys)

This notebook runs Veraxi's full **4-stage resumable evaluation pipeline** using **local open-weight MoE models** (no Gemini/OpenAI API keys):

1. **Stage 0 (CPU):** Offline Code-Intel benchmarks (`code_e2e_benchmark.py`, `code_accuracy.py`, `ts_code_accuracy.py`, `code_ablation.py`).
2. **Stage 1 (GPU, Once):** Native **Neo4j 5.x** + **Qdrant** binary + local MoE entity extraction (`fast_extraction=False`), then **snapshotted to Google Drive** so you ingest once and evaluate forever.
3. **Stage 2 (No LLM):** Pure-arithmetic retrieval eval (`vector_only` vs `hybrid_with_graph`) computing `Recall@5`, `MRR`, `Hit Rate`, latency, and context tokens per `question_type`.
4. **Stage 3a & 3b (Async vLLM + XGrammar JSON):** Decoupled **Generator** -> **Cross-Family MoE Judge** (`Qwen3-30B-A3B-Instruct-2507` -> `openai/gpt-oss-20b` with fallback) with append-only JSONL checkpoints on Google Drive.
5. **Stage 4:** Aggregates per-`question_type` tables into `BENCHMARKS.md`-ready Markdown.

> **Google AI Pro (200 CUs/mo) Tip:** Run a `--limit 20` smoke test on **L4 (~1.71 CU/hr)** first (~0.2 CUs), then switch to **A100 40GB (~5.37 CU/hr)** for the full 2,062-case run (~2–3 CUs total).


In [ ]:
#@title 1. Configuration, GPU Auto-Detection & Google Drive Mount
import os
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

# --- User Parameters ---
REPO_URL = "https://github.com/your-org/veraxi.git"  #@param {type:"string"}
REPO_BRANCH = "main"  #@param {type:"string"}
DRIVE_BASE_DIR = "/content/drive/MyDrive/veraxi-eval"  #@param {type:"string"}
LIMIT = 20  #@param {type:"integer"}  # Set 0 for all 2,062 cases
FAST_EXTRACTION = False  #@param {type:"boolean"}
FORCE_REINGEST = False  #@param {type:"boolean"}
FRESH_CHECKPOINTS = False  #@param {type:"boolean"}
JUDGE_MODE = "both"  #@param ["both", "deepeval", "deterministic"]

# --- GPU Detection & MoE Profile Selection ---
gpu_info = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
).strip()
print(f"Detected GPU: {gpu_info}")
gpu_name = gpu_info.upper()

if "A100" in gpu_name and "80GB" in gpu_name:
    PROFILE = {
        "tier": "A100-80GB",
        "gen_model": "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8",
        "gen_quant": [],
        "judge_model": "openai/gpt-oss-20b",
        "judge_fallback": "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8",
        "max_model_len": 32768,
        "kv_cache": "fp8",
        "concurrency": 24,
    }
elif "A100" in gpu_name:
    PROFILE = {
        "tier": "A100-40GB",
        "gen_model": "cyankiwi/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit",
        "gen_quant": ["--quantization", "awq"],
        "judge_model": "openai/gpt-oss-20b",
        "judge_fallback": "Qwen/Qwen3-30B-A3B-Instruct-2507-FP8",
        "max_model_len": 16384,
        "kv_cache": "fp8",
        "concurrency": 16,
    }
elif "L4" in gpu_name:
    PROFILE = {
        "tier": "L4-24GB",
        "gen_model": "cyankiwi/Qwen3-30B-A3B-Instruct-2507-AWQ-4bit",
        "gen_quant": ["--quantization", "awq"],
        "judge_model": "openai/gpt-oss-20b",
        "judge_fallback": "Qwen/Qwen2.5-14B-Instruct-AWQ",
        "max_model_len": 8192,
        "kv_cache": "fp8",
        "concurrency": 8,
    }
else:
    PROFILE = {
        "tier": "T4-16GB-Fallback",
        "gen_model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
        "gen_quant": ["--quantization", "awq"],
        "judge_model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
        "judge_fallback": "Qwen/Qwen2.5-7B-Instruct-AWQ",
        "max_model_len": 8192,
        "kv_cache": "auto",
        "concurrency": 4,
    }

print("Selected MoE Profile:", PROFILE)

DRIVE_ROOT = Path(DRIVE_BASE_DIR)
SNAPSHOT_DIR = DRIVE_ROOT / "snapshots" / "graphrag_medical"
CKPT_DIR = DRIVE_ROOT / "checkpoints"
CODE_INTEL_DIR = DRIVE_ROOT / "code_intel"
for d in (SNAPSHOT_DIR, CKPT_DIR, CODE_INTEL_DIR):
    d.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/veraxi")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(f"Working directory: {Path.cwd()}")


In [ ]:
#@title 2. Install Veraxi Backend + vLLM + DeepEval (Colab Only)
import subprocess

subprocess.run(["pip", "install", "-q", "-e", "backend[eval]", "-e", "packages/veraxi-cli"], check=True)
subprocess.run(["pip", "install", "-q", "vllm", "datasets>=5.0.1,<5.1.0", "deepeval>=4.3.4", "onnxruntime-gpu"], check=True)
print("Dependencies installed.")


In [ ]:
#@title 3. Start Native Neo4j 5.26 + Qdrant Binary (No Docker) & Restore Snapshots
import os
import subprocess
import time
import urllib.request
from pathlib import Path

NEO4J_VER = "5.26.0"
QDRANT_TAG = "v1.13.0"
SERVICES_DIR = Path("/content/services")
SERVICES_DIR.mkdir(parents=True, exist_ok=True)

# 1. Install Java 21 + Neo4j tarball
neo4j_home = SERVICES_DIR / f"neo4j-community-{NEO4J_VER}"
if not neo4j_home.exists():
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-21-jdk"], check=True)
    tgz = SERVICES_DIR / f"neo4j-community-{NEO4J_VER}-unix.tar.gz"
    if not tgz.exists():
        urllib.request.urlretrieve(
            f"https://dist.neo4j.org/neo4j-community-{NEO4J_VER}-unix.tar.gz", tgz
        )
    subprocess.run(["tar", "-xzf", str(tgz), "-C", str(SERVICES_DIR)], check=True)
    subprocess.run([str(neo4j_home / "bin/neo4j-admin"), "dbms", "set-initial-password", "veraxi_eval_pass"], check=True)

# 2. Install Qdrant binary
qdrant_bin = SERVICES_DIR / "qdrant"
qdrant_storage = SERVICES_DIR / "qdrant_storage"
qdrant_storage.mkdir(parents=True, exist_ok=True)
if not qdrant_bin.exists():
    qtgz = SERVICES_DIR / "qdrant.tar.gz"
    urllib.request.urlretrieve(
        f"https://github.com/qdrant/qdrant/releases/download/{QDRANT_TAG}/qdrant-x86_64-unknown-linux-gnu.tar.gz",
        qtgz,
    )
    subprocess.run(["tar", "-xzf", str(qtgz), "-C", str(SERVICES_DIR)], check=True)

# 3. Restore snapshots from Drive if available
neo4j_snap = SNAPSHOT_DIR / "neo4j_data.tar.gz"
qdrant_snap = SNAPSHOT_DIR / "qdrant_storage.tar.gz"
SNAPSHOT_RESTORED = False
if not FORCE_REINGEST and neo4j_snap.exists() and qdrant_snap.exists():
    print("Restoring Neo4j + Qdrant snapshots from Google Drive...")
    subprocess.run(["tar", "-xzf", str(neo4j_snap), "-C", str(neo4j_home)], check=True)
    subprocess.run(["tar", "-xzf", str(qdrant_snap), "-C", str(SERVICES_DIR)], check=True)
    SNAPSHOT_RESTORED = True

# 4. Start Neo4j and Qdrant
subprocess.run([str(neo4j_home / "bin/neo4j"), "start"], check=True)
qdrant_log = open("/content/qdrant.log", "w")
QDRANT_PROC = subprocess.Popen(
    [str(qdrant_bin)],
    cwd=str(SERVICES_DIR),
    stdout=qdrant_log,
    stderr=subprocess.STDOUT,
)
time.sleep(5)

os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USER"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "veraxi_eval_pass"
os.environ["QDRANT_URL"] = "http://localhost:6333"
os.environ["QDRANT_API_KEY"] = ""
os.environ["LLM_BASE_URL"] = "http://localhost:8000/v1"
os.environ["LLM_API_KEY"] = "local"
os.environ["LLM_MODEL_NAME"] = PROFILE["gen_model"]
os.environ["INGESTION_PROFILE"] = "lite"
print(f"Services up. SNAPSHOT_RESTORED={SNAPSHOT_RESTORED}")


In [ ]:
#@title 4. Stage 0: Offline Code-Intel Benchmarks (CPU)
import json
import subprocess

e2e = subprocess.check_output(["python", "-m", "backend.evaluation.code_e2e_benchmark"], text=True)
(CODE_INTEL_DIR / "code_e2e_benchmark.json").write_text(e2e)
print("code_e2e_benchmark:", e2e)

subprocess.run(["pytest", "-q", "backend/tests/test_code_accuracy_eval.py", "backend/tests/test_code_e2e_benchmark.py"], check=True)
subprocess.run(
    [
        "python", "-m", "backend.evaluation.run_typed_ablation",
        "--dataset", "backend/evaluation/veraxi_code_dataset.json",
        "--repo-root", ".",
        "--dense-backend", "minilm",
        "--out-json", str(CODE_INTEL_DIR / "code_ablation.json"),
        "--out-markdown", str(CODE_INTEL_DIR / "code_ablation.md"),
    ],
    check=True,
)


In [ ]:
#@title 5. Helper: Start / Stop vLLM + 3-Prompt XGrammar JSON Smoke Check
import subprocess
import time
import urllib.request
from openai import OpenAI

VLLM_PROC = None

def stop_vllm():
    global VLLM_PROC
    if VLLM_PROC is not None and VLLM_PROC.poll() is None:
        VLLM_PROC.terminate()
        try:
            VLLM_PROC.wait(timeout=20)
        except subprocess.TimeoutExpired:
            VLLM_PROC.kill()
    VLLM_PROC = None

def start_vllm(model_name: str, quant_args: list[str], port: int = 8000):
    stop_vllm()
    cmd = [
        "vllm", "serve", model_name,
        "--port", str(port),
        "--gpu-memory-utilization", "0.88",
        "--max-model-len", str(PROFILE["max_model_len"]),
        "--enable-prefix-caching",
        *quant_args,
    ]
    if PROFILE["kv_cache"] != "auto":
        cmd.extend(["--kv-cache-dtype", PROFILE["kv_cache"]])
    log_f = open(f"/content/vllm_{port}.log", "w")
    proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)
    for _ in range(90):
        if proc.poll() is not None:
            raise RuntimeError(f"vLLM exited early ({proc.returncode}). Check /content/vllm_{port}.log")
        try:
            with urllib.request.urlopen(f"http://localhost:{port}/health", timeout=2) as r:
                if r.status == 200:
                    return proc
        except Exception:
            time.sleep(3)
    raise TimeoutError(f"vLLM did not become healthy on port {port}")

def smoke_check_json(base_url: str, model_name: str):
    client = OpenAI(base_url=base_url, api_key="local")
    for i in range(3):
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": f"Return JSON with ok=true and step={i}"}],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "Smoke",
                    "schema": {"type": "object", "properties": {"ok": {"type": "boolean"}, "step": {"type": "integer"}}, "required": ["ok", "step"]},
                },
            },
            temperature=0.0,
        )
        print(f"Smoke {i+1}:", resp.choices[0].message.content)

print("vLLM lifecycle helpers ready.")


In [ ]:
#@title 6. Stage 1: Generate Dataset & Ingest Once (with Drive Snapshot)
import subprocess
from pathlib import Path

subprocess.run(["python", "backend/evaluation/generate_benchmark.py", "--dataset", "graphrag_bench", "--samples", "0"], check=True)

if not SNAPSHOT_RESTORED:
    VLLM_PROC = start_vllm(PROFILE["gen_model"], PROFILE["gen_quant"], port=8000)
    smoke_check_json("http://localhost:8000/v1", PROFILE["gen_model"])

    from backend.config import get_config
    from backend.ingestion.__main__ import run_ingestion

    get_config.cache_clear()
    cfg = get_config()
    corpus_text = Path("backend/tests/data/graphrag_test_corpus.txt").read_text(encoding="utf-8")
    schema = {
        "entities": ["Concept", "Observation", "Disease", "Symptom"],
        "relations": {"Concept": {"Disease": ["CAUSES", "TREATS", "RELATES_TO"]}},
    }
    run_ingestion(
        config=cfg,
        text=corpus_text,
        schema=schema,
        tenant_id="benchmark",
        fast_extraction=FAST_EXTRACTION,
        chunk_size=1000,
        chunk_overlap=200,
    )
    print("Ingestion complete. Snapshotting Neo4j + Qdrant to Drive...")
    subprocess.run([str(neo4j_home / "bin/neo4j"), "stop"], check=True)
    subprocess.run(["tar", "-czf", str(neo4j_snap), "-C", str(neo4j_home), "data"], check=True)
    subprocess.run(["tar", "-czf", str(qdrant_snap), "-C", str(SERVICES_DIR), "qdrant_storage"], check=True)
    subprocess.run([str(neo4j_home / "bin/neo4j"), "start"], check=True)
    print("Snapshots saved to Google Drive.")
else:
    print("Skipping ingestion: using restored Drive snapshots.")


In [ ]:
#@title 7. Stage 2: Retrieval Eval (Pure Arithmetic, No LLM)
import subprocess

cmd = [
    "python", "-m", "backend.evaluation.benchmark_rag",
    "--stage", "retrieve",
    "--checkpoint-dir", str(CKPT_DIR),
    "--limit", str(LIMIT),
]
if FRESH_CHECKPOINTS:
    cmd.append("--fresh")
subprocess.run(cmd, check=True)


In [ ]:
#@title 8. Stage 3a: Resumable Async Generation (Local MoE)
import subprocess

if VLLM_PROC is None or VLLM_PROC.poll() is not None:
    VLLM_PROC = start_vllm(PROFILE["gen_model"], PROFILE["gen_quant"], port=8000)
    smoke_check_json("http://localhost:8000/v1", PROFILE["gen_model"])

subprocess.run(
    [
        "python", "-m", "backend.evaluation.benchmark_rag",
        "--stage", "generate",
        "--checkpoint-dir", str(CKPT_DIR),
        "--limit", str(LIMIT),
        "--concurrency", str(PROFILE["concurrency"]),
    ],
    check=True,
)
stop_vllm()
print("Generator vLLM stopped cleanly.")


In [ ]:
#@title 9. Stage 3b: Cross-Family MoE Judge (XGrammar JSON) + Stage 4 Aggregate
import os
import subprocess

judge_model = PROFILE["judge_model"]
if JUDGE_MODE in ("both", "deepeval"):
    try:
        VLLM_PROC = start_vllm(judge_model, [], port=8001)
        smoke_check_json("http://localhost:8001/v1", judge_model)
    except Exception as exc:
        print(f"Primary judge {judge_model} failed ({exc}); falling back to {PROFILE['judge_fallback']}")
        judge_model = PROFILE["judge_fallback"]
        fallback_quant = ["--quantization", "awq"] if "AWQ" in judge_model else []
        VLLM_PROC = start_vllm(judge_model, fallback_quant, port=8001)
        smoke_check_json("http://localhost:8001/v1", judge_model)

    os.environ["JUDGE_LLM_BASE_URL"] = "http://localhost:8001/v1"
    os.environ["JUDGE_LLM_API_KEY"] = "local"
    os.environ["JUDGE_LLM_MODEL_NAME"] = judge_model

subprocess.run(
    [
        "python", "-m", "backend.evaluation.benchmark_rag",
        "--stage", "judge",
        "--checkpoint-dir", str(CKPT_DIR),
        "--limit", str(LIMIT),
        "--concurrency", str(PROFILE["concurrency"]),
        "--judge-mode", JUDGE_MODE,
    ],
    check=True,
)
stop_vllm()

subprocess.run(
    [
        "python", "-m", "backend.evaluation.benchmark_rag",
        "--stage", "aggregate",
        "--checkpoint-dir", str(CKPT_DIR),
        "--limit", str(LIMIT),
        "--judge-mode", JUDGE_MODE,
    ],
    check=True,
)
print((CKPT_DIR / "report.md").read_text(encoding="utf-8"))
